# Gaia2 on a budget: reproduction, failure analysis, and a waiting instruction

Yating Zhu · CS329Z Paper Video, Stanford, Fall 2026

Paper: Froger et al., *Gaia2: Benchmarking LLM Agents on Dynamic and Asynchronous Environments*, ICLR 2026 ([arXiv:2602.11964](https://arxiv.org/abs/2602.11964)). Harness: `meta-agents-research-environments` (ARE) 1.2.0.

| Section | Content |
| --- | --- |
| 1 | Reproduction: 10 Execution + 10 Time scenarios from the Gaia2 validation split, Gemini 2.5 Flash and Claude Sonnet 4.5, Llama-3.3-70B judge, one run each |
| 2 | Timelines of environment events vs agent actions for selected Time scenarios |
| 3 | Judge sensitivity: the same traces re-judged with GPT-5-mini |
| 4 | Extension: one waiting instruction added to the agent's system prompt, 10 Time scenarios re-run |
| 5 | Single-scenario runs shown in the video |
| 6 | Cost and export |

Requires two Colab Secrets: `HF_TOKEN` and `OPENROUTER_API_KEY`. Outputs are written to `/content/results/`.

## Setup

In [ ]:
%%capture --no-stderr
!pip install -q "meta-agents-research-environments==1.2.0" huggingface_hub pandas requests matplotlib

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [ ]:
# Local copy of the simulated file system. Without it, ARE fetches files from the Hub one by one
# and quickly runs into the Hub's rate limit.
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="meta-agents-research-environments/gaia2_filesystem",
    repo_type="dataset",
    allow_patterns=["demo_filesystem/*", "demo_filesystem/**"],
    local_dir="/content/gaia2_fs",
)
os.environ["DEMO_FS_PATH"] = "/content/gaia2_fs/demo_filesystem"

In [ ]:
import glob, json, shlex, subprocess, time
from pathlib import Path

import pandas as pd
import requests

RESULTS_ROOT = Path("/content/results")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

FLASH = "openrouter/google/gemini-2.5-flash"
SONNET = "openrouter/anthropic/claude-sonnet-4.5"
JUDGE_MODEL = "openrouter/meta-llama/llama-3.3-70b-instruct"   # the judge used in the paper


def _stream(cmd):
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if "429" not in line:
            print(line, end="")
    proc.wait()
    print(f"\nexit code {proc.returncode}, {(time.time() - t0) / 60:.1f} min")


def run_gaia2(config, model, limit, run_name, timeout=900, max_tries=3):
    """Run the first `limit` scenarios of a Gaia2 validation config, one at a time, one run each."""
    out_dir = RESULTS_ROOT / run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = ["are-benchmark", "run",
           "--hf-dataset", "meta-agents-research-environments/gaia2",
           "--hf-config", config, "--hf-split", "validation",
           "--model", model, "--provider", "local",
           "--judge_model", JUDGE_MODEL, "--judge_provider", "local",
           "--agent", "default", "--num_runs", "1", "--limit", str(limit),
           "--max_concurrent_scenarios", "1", "--scenario_timeout", str(timeout),
           "--trace_dump_format", "both", "--output_dir", str(out_dir)]
    (out_dir / "command.sh").write_text(" ".join(shlex.quote(c) for c in cmd) + "\n")
    for attempt in range(1, max_tries + 1):
        _stream(cmd)
        if (out_dir / "benchmark_stats.json").exists():
            return out_dir
        if attempt < max_tries:
            print("No benchmark_stats.json; retrying in 5 minutes.")
            time.sleep(300)
    raise RuntimeError(f"{run_name} did not complete")


def summarize(run_names):
    rows = []
    for name in run_names:
        s = json.load(open(RESULTS_ROOT / name / "benchmark_stats.json"))
        g, m = s["statistics"]["global"], s["metadata"]
        rows.append({"run": name, "model": m.get("model", "").split("/")[-1],
                     "scenarios": g["total_runs"], "passed": g["success_runs"],
                     "success_rate_%": round(g["macro_success_rate"], 1),
                     "avg_duration_s": round(g["avg_run_duration"], 1)})
    df = pd.DataFrame(rows).set_index("run")
    display(df)
    return df


def api_usage():
    r = requests.get("https://openrouter.ai/api/v1/key",
                     headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"}, timeout=30)
    r.raise_for_status()
    return r.json()["data"]["usage"]


usage_start = api_usage()

## 1. Reproduction

In [ ]:
run_gaia2("execution", FLASH, limit=10, run_name="exec_flash")

In [ ]:
run_gaia2("time", FLASH, limit=10, run_name="time_flash")

In [ ]:
run_gaia2("execution", SONNET, limit=10, run_name="exec_strong")

In [ ]:
run_gaia2("time", SONNET, limit=10, run_name="time_strong")

In [ ]:
summarize(["exec_flash", "exec_strong", "time_flash", "time_strong"])

## 2. Timelines: environment events vs agent actions

Top row: environment events (red ▼; faded if scheduled but never reached). Bottom row: agent actions (grey bar = waiting, green = write action, orange = report to user) and the times at which the oracle expected a write (hollow green ▲). Figures are saved to `results/figs/`.

In [ ]:
import json, glob
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt

FIG_DIR = Path("/content/results/figs"); FIG_DIR.mkdir(parents=True, exist_ok=True)

SHORT = {
    "send_message": "send message", "send_message_to_group_conversation": "send group msg",
    "send_message_to_user": "report to user", "wait_for_notification": "wait",
    "search_apartments": "search", "save_apartment": "save apt", "remove_saved_apartment": "unsave apt",
    "add_new_apartment": "new listing", "create_and_add_message": "reply arrives",
    "add_calendar_event": "add calendar", "send_email": "send email",
}
def _fn(a): return (a or {}).get("function") or ""
def _name(a): return SHORT.get(_fn(a), _fn(a) or "?")
def _arg(a, key):
    for x in (a or {}).get("args") or []:
        if isinstance(x, dict) and x.get("name") == key: return str(x.get("value"))
    return ""

def load_trace(path):
    d = json.load(open(path))
    done = [e for e in d["completed_events"] if e.get("event_time") is not None]
    t0 = min(e["event_time"] for e in done)
    agent = [(e["event_time"] - t0, e["action"]) for e in done if e["event_type"] == "AGENT"]
    env = [(e["event_time"] - t0, e["action"]) for e in done if e["event_type"] == "ENV"]
    happened = {e["event_id"] for e in done}
    env_time = {e["event_id"]: e["event_time"] - t0 for e in done if e["event_type"] == "ENV"}
    missed_env = []
    for e in d["events"]:
        if e["event_type"] == "ENV" and e["event_id"] not in happened:
            missed_env.append((e.get("event_relative_time") or 0, e["action"]))
            env_time.setdefault(e["event_id"], e.get("event_relative_time") or 0)
    expected = []
    for e in d["events"]:
        if e["class_name"] != "OracleEvent": continue
        deps = [x for x in (e.get("dependencies") or []) if isinstance(x, str) and x.startswith("Event-ENV")]
        if deps and _fn(e["action"]) in ("send_message","send_email","remove_saved_apartment","add_calendar_event","send_message_to_group_conversation"):
            expected.append((env_time.get(deps[0], 0) + (e.get("event_relative_time") or 0), e["action"]))
    meta = d["metadata"]
    ann = meta.get("annotation") if isinstance(meta.get("annotation"), dict) else {}
    return dict(agent=agent, env=env, missed_env=missed_env, expected=expected,
                decision=ann.get("validation_decision"), model=(meta.get("simulation") or {}).get("model_id",""),
                sid=(meta.get("definition") or {}).get("scenario_id", Path(path).stem))

def plot_timeline(path, title=None, xmax=None, save_as=None):
    r = load_trace(path)
    fig, ax = plt.subplots(figsize=(13, 5))
    for i, (t, a) in enumerate(sorted(r["env"], key=lambda x: x[0])):
        ax.plot([t], [1], marker="v", ms=11, color="#d62728", ls="none")
        price = _arg(a, "price")
        label = (_arg(a, "name") or _arg(a, "content")[:18]) + (f"  ${float(price):.0f}" if price not in ("", "None") else "")
        ax.annotate(f"{_name(a)}\n{label}", (t, 1), xytext=(0, 12 + 22 * (i % 3)), textcoords="offset points", ha="center", fontsize=8, color="#d62728")
    for t, a in r["missed_env"]:
        ax.plot([t], [1], marker="v", ms=11, color="#d62728", ls="none", alpha=0.35)
        ax.annotate(f"scripted, never reached:\n{_name(a)} at +{t:.0f}s", (t, 1), xytext=(0, 12), textcoords="offset points", ha="center", fontsize=8, color="#d62728", alpha=0.8)
    for t, a in r["expected"]:
        ax.plot([t], [0], marker="^", ms=10, mfc="none", mec="#2ca02c", mew=1.6, ls="none")
        ax.annotate(f"expected: {_name(a)}", (t, 0), xytext=(0, 14), textcoords="offset points", ha="center", fontsize=8, color="#2ca02c")
    last_t = 0
    for t, a in r["agent"]:
        fn = _fn(a); color, marker = "#1f77b4", "o"
        if fn == "wait_for_notification":
            color, marker = "#7f7f7f", "s"
            ax.plot([t, t + float(_arg(a, "timeout") or 0)], [0, 0], color="#7f7f7f", lw=6, alpha=0.35, solid_capstyle="butt")
        if fn in ("send_message","send_message_to_group_conversation","send_email","remove_saved_apartment","add_calendar_event"): color = "#2ca02c"
        if fn == "send_message_to_user": color = "#ff7f0e"
        ax.plot([t], [0], marker=marker, ms=8, color=color, ls="none")
        if fn in ("send_message","send_message_to_group_conversation","send_email","remove_saved_apartment","add_calendar_event","send_message_to_user"):
            ax.annotate(_name(a), (t, 0), xytext=(0, -22), textcoords="offset points", ha="center", fontsize=8, color=color)
        last_t = max(last_t, t)
    ax.axvline(last_t, color="#ff7f0e", ls="--", lw=1)
    ax.text(last_t, 0.5, f"  agent stops at t+{last_t:.0f}s", color="#ff7f0e", fontsize=9, va="center")
    ax.set_yticks([0, 1]); ax.set_yticklabels(["what the agent did", "what happened in the world"])
    ax.set_xlabel("simulated time (seconds)")
    allt = [t for t, _ in r["agent"]] + [t for t, _ in r["env"]] + [t for t, _ in r["missed_env"]] + [t for t, _ in r["expected"]]
    ax.set_xlim(-5, xmax or (max(allt) + 30)); ax.set_ylim(-0.8, 2.3); ax.grid(axis="x", alpha=0.3)
    ax.set_title(title or f"{r['sid']} | {r['model'].split('/')[-1]} | verdict: {r['decision']}", fontsize=11)
    fig.tight_layout()
    out = FIG_DIR / (save_as or f"{r['sid']}_{r['model'].split('/')[-1]}.png")
    fig.savefig(out, dpi=160); plt.show()
    print("saved", out)
    return r

def trace_path(run, sid):
    hits = glob.glob(f"/content/results/{run}/hf/{sid}*.json")
    assert hits, f"not found: {run}/{sid}"
    return hits[0]

In [ ]:
for run in ["time_strong", "time_flash"]:
    plot_timeline(trace_path(run, "scenario_universe_21_xvc7uo"), xmax=270)
for run in ["time_flash", "time_strong"]:
    plot_timeline(trace_path(run, "scenario_universe_21_wezjyh"), xmax=200)

## 3. Judge sensitivity
The saved traces are re-judged with GPT-5-mini; the agents are not re-run.

In [ ]:
import subprocess, shlex, time, json, glob
from pathlib import Path
import pandas as pd

NEW_JUDGE = "openrouter/openai/gpt-5-mini"

def rejudge(run_name, new_judge=NEW_JUDGE, out_suffix="rejudge", concurrency=5, timeout=600):
    src_dir = Path(f"/content/results/{run_name}/hf")
    out_dir = Path(f"/content/results/{run_name}_{out_suffix}"); out_dir.mkdir(parents=True, exist_ok=True)
    cmd = ["are-benchmark", "judge", "-d", str(src_dir),
           "--model", "openrouter/google/gemini-2.5-flash", "--provider", "local",   # required arg; not called when only judging
           "--judge_model", new_judge, "--judge_provider", "local",
           "--max_concurrent_scenarios", str(concurrency), "--scenario_timeout", str(timeout),
           "--trace_dump_format", "both", "--output_dir", str(out_dir)]
    print("Command:\n  " + " ".join(shlex.quote(c) for c in cmd) + "\n")
    (out_dir / "command.sh").write_text(" ".join(shlex.quote(c) for c in cmd) + "\n")
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "429" in line or "lazy load" in line: continue
        if any(k in line for k in ("Result:", "Validation Report", "success rate", "ERROR", "Traceback", "Error")):
            print(line, end="")
    p.wait(); print(f"\nexit code {p.returncode}, {(time.time()-t0)/60:.1f} min. Results in {out_dir}")
    return out_dir

def decisions(run_name):
    out = {}
    for fp in glob.glob(f"/content/results/{run_name}/**/lite/*.json", recursive=True):
        d = json.load(open(fp)); out[d["scenario_id"]] = d.get("validation_decision")
    return out

def compare_judges(run_name, suffix="rejudge"):
    a, b = decisions(run_name), decisions(f"{run_name}_{suffix}")
    rows = [{"scenario": s, "Llama-3.3-70B judge": a.get(s), "GPT-5-mini judge": b.get(s),
             "differs": (a.get(s) != b.get(s))} for s in sorted(a)]
    df = pd.DataFrame(rows)
    n_valid_old = sum(v == "Valid" for v in a.values()); n_valid_new = sum(v == "Valid" for v in b.values())
    print(f"{run_name}: Llama-3.3-70B {n_valid_old}/{len(a)} valid, {NEW_JUDGE.split('/')[-1]} {n_valid_new}/{len(b)} valid, {int(df['differs'].sum())} verdicts differ")
    display(df)
    return df

In [ ]:
for run in ["time_flash", "exec_strong"]:
    rejudge(run)
    compare_judges(run)

## 4. Extension: an explicit waiting instruction

The paragraph below is inserted into the default agent's system prompt (through the `json_agent_hints` slot of ARE's prompt template). The 10 Time scenarios are then re-run with Gemini 2.5 Flash, and the prompt is restored.

In [ ]:
import shutil
import are.simulation.agents.default_agent.prompts.system_prompt as sp

WAITING_HINT = """
TIME-SENSITIVE TASKS: If the user's request depends on something that may happen later (for example "if she replies within 4 minutes",
"in exactly N minutes", "each time a new listing appears"), do NOT end your turn after finishing the first part.
Instead: (1) call SystemApp__get_current_time to note the time, (2) call SystemApp__wait_for_notification with a short timeout
(30 seconds or less, so you can react quickly), (3) when it returns, proactively check the relevant app for new messages or changes
(environment changes are NOT pushed to you), (4) act on what you find, and repeat until the time window has passed.
Only send your final report to the user after every time-dependent part of the task has been handled."""

PROMPT_FILE = Path(sp.__file__)
BACKUP = PROMPT_FILE.with_suffix(".py.bak")
if not BACKUP.exists():
    shutil.copy(PROMPT_FILE, BACKUP)


def patch_prompt():
    src = BACKUP.read_text()
    anchor = "DEFAULT_ARE_SIMULATION_REACT_JSON_SYSTEM_PROMPT = SYSTEM_PROMPT_TEMPLATE.format("
    i = src.index(anchor)
    j = src.index('json_agent_hints="",', i)
    PROMPT_FILE.write_text(src[:j] + "json_agent_hints=" + repr(WAITING_HINT) + "," + src[j + len('json_agent_hints="",'):])


def restore_prompt():
    shutil.copy(BACKUP, PROMPT_FILE)

In [ ]:
patch_prompt()
try:
    run_gaia2("time", FLASH, limit=10, run_name="time_flash_wait")
finally:
    restore_prompt()
summarize(["time_flash", "time_flash_wait"])

In [ ]:
plot_timeline(trace_path("time_flash_wait", "scenario_universe_21_wezjyh"), xmax=260)

## 5. Single-scenario runs

Scenario `wezjyh`: save three apartments, message a housemate, wait up to four minutes for her reply, then act on it. The scenario is copied into a one-scenario local dataset so that the harness runs only this one.

In [ ]:
import textwrap
from datasets import load_dataset

WEZ = "scenario_universe_21_wezjyh"
SINGLE_ROOT = Path("/content/single")
WRITES = {"send_message", "send_message_to_group_conversation", "send_email", "remove_saved_apartment",
          "add_calendar_event", "save_apartment", "send_message_to_user"}


def prepare_single(sid, config="time"):
    d = SINGLE_ROOT / sid / config / "validation"
    if not list(d.glob("*.json")):
        for row in load_dataset("meta-agents-research-environments/gaia2", name=config,
                                split="validation", streaming=True):
            if row.get("scenario_id") == sid:
                d.mkdir(parents=True, exist_ok=True)
                data = row["data"]
                (d / f"{sid}.json").write_text(data if isinstance(data, str) else json.dumps(data))
                break
        else:
            raise ValueError(f"scenario not found: {sid}")
    return SINGLE_ROOT / sid


def run_one(sid, tag, config="time", model=FLASH, hint=False, timeout=600):
    root = prepare_single(sid, config)
    out_dir = RESULTS_ROOT / f"single_{tag}_{time.strftime('%m%d_%H%M%S')}"
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = ["are-benchmark", "run", "--dataset", str(root), "--config", config, "--split", "validation",
           "--model", model, "--provider", "local",
           "--judge_model", JUDGE_MODEL, "--judge_provider", "local",
           "--agent", "default", "--num_runs", "1", "--max_concurrent_scenarios", "1",
           "--scenario_timeout", str(timeout), "--trace_dump_format", "both", "--output_dir", str(out_dir)]
    (out_dir / "command.sh").write_text(" ".join(shlex.quote(c) for c in cmd) + "\n")
    print(f"agent: {model.split('/')[-1]} | system prompt: {'default + waiting instruction' if hint else 'default'}")
    if hint:
        patch_prompt()
    t0 = time.time()
    try:
        with open(out_dir / "run.log", "w") as log:
            p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in p.stdout:
                log.write(line)
                print(f"\rrunning... {time.time() - t0:5.0f} s", end="")
            p.wait()
    finally:
        restore_prompt()
    print(f"\rfinished in {time.time() - t0:.0f} s" + " " * 20)
    hits = glob.glob(f"{out_dir}/**/hf/{sid}*.json", recursive=True)
    if not hits:
        print("".join(open(out_dir / "run.log").readlines()[-30:]))
        raise RuntimeError("no trace produced")
    return hits[0]


def _args(action):
    return {x["name"]: x.get("value") for x in (action or {}).get("args") or [] if isinstance(x, dict)}


def show_task(trace):
    d = json.load(open(trace))
    user = next(e for e in d["events"] if e["event_type"] == "USER")
    print("USER REQUEST")
    print(textwrap.fill(_args(user["action"]).get("content", ""), 110))
    print("\nSCHEDULED ENVIRONMENT EVENTS (not visible to the agent)")
    for e in d["events"]:
        if e["event_type"] == "ENV":
            print(f"  +{e.get('event_relative_time') or 0:>4.0f} s  reply: \"{str(_args(e['action']).get('content', ''))[:90]}\"")
    print("\nORACLE WRITE ACTIONS")
    for e in d["events"]:
        if e["class_name"] == "OracleEvent":
            print("  -", e["action"]["function"])


def show_steps(trace):
    d = json.load(open(trace))
    done = sorted((e for e in d["completed_events"] if e.get("event_time") is not None), key=lambda e: e["event_time"])
    t0 = done[0]["event_time"]
    for e in done:
        a = e.get("action") or {}
        fn, t = a.get("function") or "", e["event_time"] - t0
        if e["event_type"] == "AGENT":
            note = _args(a).get("content") or _args(a).get("timeout") or ""
            print(f"  t+{t:5.0f}s  agent  {fn:<38}{'WRITE' if fn in WRITES else '':<7}{str(note)[:60]}")
        elif e["event_type"] == "ENV":
            print(f"  t+{t:5.0f}s  env    reply: \"{str(_args(a).get('content', ''))[:70]}\"")
    reached = {e["event_id"] for e in done}
    for e in d["events"]:
        if e["event_type"] == "ENV" and e["event_id"] not in reached:
            print(f"  not reached: env reply scheduled at +{e.get('event_relative_time') or 0:.0f}s")
    print("\nverdict:", (d["metadata"].get("annotation") or {}).get("validation_decision"))


def show_email(run, key):
    trace = glob.glob(str(RESULTS_ROOT / run / "hf" / f"*{key}*.json"))[0]
    d = json.load(open(trace))
    sent = [e["action"] for e in d["completed_events"]
            if e["event_type"] == "AGENT" and e["action"].get("function") == "send_email"]
    expected = [e["action"] for e in d["events"]
                if e["class_name"] == "OracleEvent" and e["action"].get("function") == "send_email"]
    for title, actions in (("SENT BY AGENT", sent), ("EXPECTED BY ORACLE", expected)):
        print(f"===== {title} =====")
        for a in actions:
            x = _args(a)
            print("to:", x.get("recipients"), "| subject:", x.get("subject"))
            print(textwrap.fill(str(x.get("content", "")), 110), "\n")
    lite = glob.glob(str(RESULTS_ROOT / run / "lite" / f"*{key}*.json"))
    if lite:
        l = json.load(open(lite[0]))
        print("verdict:", l.get("validation_decision"), "|", (l.get("validation_rationale") or "")[:300])

In [ ]:
show_task(trace_path("time_flash", WEZ))

In [ ]:
t_default = run_one(WEZ, "default")
show_steps(t_default)
plot_timeline(t_default, xmax=200)

In [ ]:
print(WAITING_HINT)
t_wait = run_one(WEZ, "wait", hint=True)
show_steps(t_wait)
plot_timeline(t_wait, xmax=260)

Scenario `2tof7f` (Execution, Claude Sonnet 4.5): the email the agent sent vs the email the oracle expects.

In [ ]:
show_email("exec_strong", "2tof7f")

## 6. Cost and export

In [ ]:
print(f"OpenRouter usage for this notebook: ${api_usage() - usage_start:.2f}")
summary = summarize(["exec_flash", "exec_strong", "time_flash", "time_strong", "time_flash_wait"])
(RESULTS_ROOT / "figs").mkdir(exist_ok=True)
summary.to_csv(RESULTS_ROOT / "figs" / "summary.csv")

In [ ]:
zip_path = shutil.make_archive("/content/gaia2_results", "zip", RESULTS_ROOT)
from google.colab import files
files.download(zip_path)